<a href="https://colab.research.google.com/github/syedmahmoodiagents/NLP/blob/main/ShortFullTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Block(nn.Module):
    def __init__(self, embedding_dim, block_size, n_heads):
        super().__init__()
        # Replace custom MultiHeadAttention with nn.MultiheadAttention
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=n_heads,
            batch_first=True  # Use batch-first format for easier tensor handling
        )
        self.ffwd = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        # Create causal mask for autoregressive attention
        self.register_buffer('causal_mask', torch.triu(torch.ones(block_size, block_size), diagonal=1).bool())

    def forward(self, x):
        # Self-attention with causal masking
        B, T, C = x.shape
        mask = self.causal_mask[:T, :T]  # Get the appropriate sized mask

        attn_out, _ = self.attention(
            self.ln1(x),
            self.ln1(x),
            self.ln1(x),
            attn_mask=mask
        )
        x = x + attn_out

        # Feed-forward
        x = x + self.ffwd(self.ln2(x))
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embedding_dim, block_size, n_heads, n_layers):
        super().__init__()
        self.token_embedding_table = nn.Embedding(src_vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Block(embedding_dim, block_size, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, tgt_vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss